# 04 - Preprocessing: outliers and missing values

Use `train_clean.csv` from `01_data_cleaning.ipynb` to review extreme values, compare spending-target transformations and inspect missing predictors after repairs.

Fit any imputation, encoding, scaling or percentile limits inside the model pipeline, separately on each temporal training fold.

In [1]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy.stats import spearmanr

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

import src.data as data_module

importlib.reload(data_module)
from src.data import load_clean

DATA_DIR = project_root / "data"
processed_dir = DATA_DIR / "processed"

TARGET, SPEND = "repeat_purchase_90d", "spend_next_90d"
BLUE, ORANGE, GREY, RED = "#2f6690", "#d9822b", "#9aa5b1", "#c0392b"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Same plot style as notebook 01.
pio.templates["eda"] = go.layout.Template(
    layout=dict(
        font=dict(size=13),
        title=dict(font=dict(size=17)),
        height=450,
        margin=dict(t=80, l=60, r=30, b=50),
        hoverlabel=dict(bgcolor="white"),
    )
)
pio.templates.default = "plotly_white+eda"

# Transaction files saved by notebook 01.
df = load_clean(str(processed_dir / "train_clean.csv"))
test = load_clean(str(processed_dir / "test_clean.csv"))

# Customer lookup saved by notebook 01b.
customers = pd.read_csv(
    processed_dir / "clientes_clean.csv",
    usecols=["customer_id", "country_or_region", "island"],
).set_index("customer_id")
assert customers.index.is_unique, "clientes_clean.csv has a repeated customer_id"

print(
    "Train:", df.shape,
    "| Test:", test.shape,
    "| Customer profiles:", len(customers),
)

Train: (6469, 40) | Test: (986, 38) | Customer profiles: 1143


## 1. Outliers in the features

Two questions for each numeric column: **how many values are extreme**, and **are they real or errors**?

First the distribution of each column (choose it in the dropdown), then the number of rows outside the IQR limits: below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR (the box-plot rule).

In [2]:
from plotly.subplots import make_subplots

num_cols = df.select_dtypes(include="number").columns.drop(
    [
        "ID",
        "customer_id",
        "repeat_purchase_90d",
        "spend_next_90d",
        "random_noise",
    ],
    errors="ignore",
)

fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3], vertical_spacing=0.15,
                    subplot_titles=("Histogram of the middle 98% (1%-99%)", "Box plot of all values"))
buttons = []
for i, variable in enumerate(num_cols):
    values = df[variable].dropna()
    low, high = values.quantile([0.01, 0.99])
    bulk = values[values.between(low, high)]
    fig.add_histogram(x=bulk, nbinsx=50, marker_color=BLUE, name=variable, visible=(i == 0), row=1, col=1)
    fig.add_box(x=values, marker_color=BLUE, boxpoints="outliers", name="", visible=(i == 0), row=2, col=1)

    title = (f"{variable}<br><sup>median {values.median():,.2f} | max {values.max():,.2f} | "
             f"missing {df[variable].isna().sum()} | outside 1%-99%: {len(values) - len(bulk)}</sup>")
    visible = [False] * (2 * len(num_cols))
    visible[2 * i] = visible[2 * i + 1] = True
    buttons.append(dict(label=variable, method="update", args=[{"visible": visible}, {"title.text": title}]))

fig.update_layout(title=buttons[0]["args"][1]["title.text"], height=600, showlegend=False,
                  updatemenus=[dict(buttons=buttons, x=1, xanchor="right", y=1.2, yanchor="top")])
fig.update_yaxes(title_text="number of rows", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.show()

In [3]:
# Rows outside the IQR limits, per column
rows = {}
for col in num_cols:
    values = df[col].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    below = (values < q1 - 1.5 * iqr).sum()
    above = (values > q3 + 1.5 * iqr).sum()
    rows[col] = {"below lower limit": below, "above upper limit": above,
                 "% outliers": round(100 * (below + above) / len(values), 1)}
iqr_table = pd.DataFrame(rows).T.astype({"below lower limit": int, "above upper limit": int})
iqr_table.sort_values("% outliers", ascending=False)

,below lower limit,above upper limit,% outliers
spend_30d,0,1060,16.4
Total Portes,0,1053,16.3
Total Desc_ Linha,0,822,12.7
prior_min_purchase_value,0,671,11.4
purchase_count_180d,0,737,11.4
spend_90d,0,661,10.2
prior_total_spend,0,642,9.9
Total a Pagar,0,592,9.2
prior_std_gap_days,0,461,9.1
Total Liquido,0,587,9.1


### Key insight - Section 1

**Found**

- In the cleaned training data, the IQR flags occur above the upper limit. Monetary and purchase-count columns have long right tails.
- When the IQR is zero, any value different from the quartiles is flagged. That is why the rule flags all 1,053 purchases with shipping in `Total Portes`.
- Notebook 01 documents the verified errors and repairs. The IQR flags here identify values for inspection.

**For modelling**

- Start with the original cleaned values, including plausible extreme purchases.
- For models sensitive to feature scale, compare scaling alone with `log1p` followed by scaling on selected nonnegative, right-skewed features. Any percentile capping evaluated later should learn its bounds within the training fold.

### What would percentile capping change?

Capping replaces values above a chosen limit with that limit. It can reduce the influence of extreme values, but it also removes differences between large observations. We inspect its effect before deciding where to apply it.

We reproduce the preprocessing steps that occur before capping: clean the transactions, create the candidate features, and fill gaps caused by insufficient purchase history. We then calculate the training 99th percentile for each retained numeric feature.

The table shows:

- **Cap:** the hypothetical upper limit estimated from training data.
- **Train maximum:** the largest value before capping.
- **Train rows above the cap:** the percentage that would change if this feature were capped.
- **Test rows above the cap:** how often later purchases exceed the training limit.
- **Capped by current config:** whether capping is actually enabled for that feature through `cap_columns`.

The limits are estimated from **training data only**. Test data helps us inspect their effect without determining the thresholds. A large proportion of test values above a limit may reflect changes over time rather than incorrect values.

Because structural fills are already present, artificial values such as **4,000 days** contribute to these percentiles. These thresholds describe the inputs reaching the capping step; they should not be interpreted as percentiles of observed customer behaviour alone.

This cell only inspects potential changes. Final capping choices should be compared through temporal validation, estimating limits separately within each training fold.

In [4]:
# Effect of capping: the data exactly as the pipeline sees it just before the capping step
from src.cleaning import clean_transactions
from src.data import add_candidate_features
from src.preprocessing import fill_no_history, load_config

all_config = load_config(project_root / "config.yaml")
config = all_config["preprocessing"]
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")


def before_capping(raw, is_train):
    """clean_transactions -> add_candidate_features -> fill_no_history, as in the pipeline."""
    data = clean_transactions(raw, history=train_raw, is_train=is_train)
    data = add_candidate_features(data, customers.reset_index())
    return fill_no_history(data, config["no_history_days"], config["no_history_ratio"])


train_pipe = before_capping(train_raw, is_train=True)
test_pipe = before_capping(test_raw, is_train=False)

# Hypothetical 99th-percentile caps for every numeric feature; the final column shows which are actually enabled.
# The numeric features left after drop_correlated (as build_preprocessor does)
numeric = [col for col in config["numeric_features"] if col not in config["drop_correlated"]]
cap = train_pipe[numeric].quantile(config["cap_quantiles"][1])

capping = pd.DataFrame({
    "cap (train 99th pct)": cap,
    "train max": train_pipe[numeric].max(),
    "% train rows changed": 100 * (train_pipe[numeric] > cap).mean(),
    "% test rows above cap": 100 * (test_pipe[numeric] > cap).mean(),
}).round(2)
cap_columns = config.get("cap_columns")
if cap_columns is None:
    cap_columns = numeric
capping["capped by current config"] = capping.index.isin(cap_columns)
display(capping.sort_values("% train rows changed", ascending=False))


,cap (train 99th pct),train max,% train rows changed,% test rows above cap,capped by current config
Total a Pagar,6543.97,22297.21,1.00,1.83,True
Total Imp_,1026.79,4169.40,1.00,2.43,True
Total Desc_ Global,7.85,218.49,1.00,1.72,True
Total Desc_ Linha,1369.04,4103.44,1.00,3.75,True
Custo,2382.10,7427.40,1.00,1.83,True
prior_total_spend,158299.74,247865.10,1.00,8.01,True
prior_mean_purchase_value,3809.53,6685.17,1.00,2.54,True
prior_purchase_count,211.32,276.00,1.00,4.26,False
spend_365d,34611.57,75013.40,1.00,1.72,True
prior_std_gap_days,426.77,2212.54,1.00,2.33,False


### Key insight - effect of capping

- The table shows hypothetical caps for all numeric features and identifies the columns actually capped by the current configuration. Bounds are fitted on training rows only.
- About one third of test purchases exceed the hypothetical training tenure cap. Tenure grows over time; crossing that limit does not establish an error.
- The revised candidate policy caps selected monetary features. It leaves tenure, purchase counts, tax rates, binary flags, noise and structural gap sentinels uncapped. Compare this policy with no capping on the same temporal folds before choosing the final policy.
- Capping can affect both linear models and trees. It merges distinct values above the bound, so a tree can no longer distinguish those values through that feature.
- The original 4,000-day and gap-ratio-10 fills remain configurable candidate encodings. They indicate unavailable history; they are not observed durations or ratios. Compare them with explicit history-availability indicators during model optimization.


## 2. Outliers in the target `spend_next_90d`

Analyse purchases followed by another purchase within 90 days that also have a usable spending target. Large prediction errors have more influence on RMSE because errors are squared. Squared deviations from the mean show how much the largest targets contribute to the in-sample error of a constant-mean baseline.

In [5]:
returned = df.loc[df[TARGET].eq(1) & df[SPEND].notna()]
spend = returned[SPEND]
p99 = spend.quantile(0.99)
top = spend.ge(p99)
squared_dev = (spend - spend.mean()) ** 2

print(spend.describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_string())
print(f"\nAt or above the 99th percentile: {top.sum()} rows "
      f"from {returned.loc[top, 'customer_id'].nunique()} customers")
print(f"  Share of total spending: {spend[top].sum() / spend.sum():.1%}")
print(f"  Share of squared deviations from the mean: "
      f"{squared_dev[top].sum() / squared_dev.sum():.1%}")
print(f"  Median spend_365d, top group vs rest: "
      f"{returned.loc[top, 'spend_365d'].median():,.0f} vs "
      f"{returned.loc[~top, 'spend_365d'].median():,.0f}")

count     3560.0
mean      1874.0
std       2541.0
min          1.0
50%        939.0
90%       4976.0
99%      12092.0
max      25654.0

At or above the 99th percentile: 36 rows from 5 customers
  Share of total spending: 8.6%
  Share of squared deviations from the mean: 32.9%
  Median spend_365d, top group vs rest: 33,716 vs 4,349


In [6]:
fig = go.Figure()
fig.add_scatter(x=np.log1p(returned.loc[~top, "spend_365d"]), y=np.log1p(spend[~top]), mode="markers",
                marker=dict(size=5, color=BLUE, opacity=0.3), name="other returners")
fig.add_scatter(x=np.log1p(returned.loc[top, "spend_365d"]), y=np.log1p(spend[top]), mode="markers",
                marker=dict(size=9, color=RED), name="top 1% spend")
fig.update_layout(title="The biggest spenders were already big spenders", height=550,
                  xaxis_title="log(1 + spend_365d)", yaxis_title="log(1 + spend_next_90d)")
fig.show()

In [7]:
target_options = pd.DataFrame(
    {
        "skewness": [spend.skew(), np.log1p(spend).skew(), spend.clip(upper=p99).skew()],
        "rows capped": [0, 0, spend.gt(p99).sum()],
    },
    index=["raw", "log1p", f"capped at the 99th percentile ({p99:,.0f})"],
)
display(target_options.round(2))

,skewness,rows capped
raw,3.08,0
log1p,-0.55,0
"capped at the 99th percentile (12,092)",2.18,36


### Key insight - Section 2

**Found**

- Median observed spending is about €940 and the maximum is about €25,600.
- The 36 purchases at or above the 99th percentile come from five customers. They represent about 9% of total spending and 33% of the squared deviations from the overall mean.
- Their median `spend_365d` is about €33,700, compared with €4,300 for the rest. That makes prior spending worth testing as a predictor of high future spending.
- Skewness is 3.08 on the original scale, −0.55 after `log1p` and 2.18 after capping at the 99th percentile.

**For modelling**

- Keep the observed spending values as the regression target. Compare training on euros with training on `log1p(spend)`.
- Use `expm1` to return log-model predictions to euros. Compare both models on the same temporal validation folds using RMSE, MAE and R² on the original scale.

## 3. Missing predictors after cleaning

Count the missing values in the cleaned training features. Compare selected history features by number of prior purchases to distinguish fields that are undefined with limited history from gaps that occur despite having history. The four feature columns in the second table show percentages within each purchase-count group.

In [8]:
# Every column the model could use: everything except identifiers, the date and the two targets
predictor_cols = df.columns.drop(["ID", "customer_id", "purchase_date", TARGET, SPEND], errors="ignore")

missing = df[predictor_cols].isna().sum()
missing_summary = pd.DataFrame({
    "missing rows": missing,
    "missing (%)": missing.div(len(df)).mul(100).round(2),
})
display(
    missing_summary.loc[missing.gt(0)]
    .sort_values("missing rows", ascending=False)
)

,missing rows,missing (%)
prior_std_gap_days,1413,21.84
prior_std_purchase_value,1048,16.20
prior_mean_gap_days,1048,16.20
days_since_previous_purchase,603,9.32
prior_min_purchase_value,603,9.32
prior_mean_purchase_value,603,9.32
prior_max_purchase_value,603,9.32
spend_365d,200,3.09
transport_method,196,3.03
payment_type,195,3.01


In [9]:
# Every column that needs past purchases, from the same list fill_no_history uses
# (gap_ratio is left out: it is only created later, by add_candidate_features)
from src.preprocessing import NEEDS_PAST_PURCHASES

history_features = [col for col in NEEDS_PAST_PURCHASES if col != "gap_ratio"]
prior_count_group = pd.cut(
    df["prior_purchase_count"],
    bins=[0, 1, 2, 3, np.inf],
    labels=["0", "1", "2", "3+"],
    right=False,
)
missing_by_history = (
    df[history_features].isna()
    .groupby(prior_count_group, observed=True)
    .mean()
    .mul(100)
    .round(1)
)
missing_by_history.insert(0, "rows", prior_count_group.value_counts(sort=False))
display(missing_by_history.rename_axis("Prior purchases"))

,rows,days_since_previous_purchase,prior_min_purchase_value,prior_mean_purchase_value,prior_max_purchase_value,prior_std_purchase_value,prior_mean_gap_days,prior_std_gap_days
Prior purchases,,,,,,,,
0,603,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,445,0.0,0.0,0.0,0.0,100.0,100.0,100.0
2,365,0.0,0.0,0.0,0.0,0.0,0.0,100.0
3+,5056,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Does a structural gap carry information?

If customers without enough history behave differently, the gap itself is information. Filling it with a typical value would hide it (class 10).

In [10]:
# Repeat rate by number of past purchases: < 2 past purchases means gap_ratio and the mean gap don't exist
history_group = np.where(df["prior_purchase_count"] < 2, "fewer than 2 past purchases", "2 or more past purchases")
df.groupby(history_group)[TARGET].agg(["mean", "size"]).round(3).rename(columns={"mean": "repeat rate", "size": "rows"})

,repeat rate,rows
2 or more past purchases,0.60,5421
fewer than 2 past purchases,0.35,1048


The structural gap carries information. Customers with fewer than 2 past purchases (1,048 rows, about 16% of the data) have a repeat rate of 0.35, against 0.60 for customers with 2 or more past purchases (5,421 rows). That is a difference of 25 percentage points, and both groups are large enough for it not to be noise.

Because `gap_ratio` and the mean gap cannot be computed for the first group, filling them with a typical value (median or KNN) would erase the signal that separates a low-repeat group from a high-repeat one. These columns are missing for a structural reason (not enough history), not at random.

**Decision.** Keep an explicit history-availability flag (for example `has_enough_history`, or `prior_purchase_count < 2`) as a model input, and fill the gap features separately from it. The flag preserves the information, and the imputed value only keeps the row usable. This also replaces the artificial "no previous purchase" sentinel values, which the flag makes unnecessary.

**Caveat.** This is a raw comparison and not a causal effect. Customers with fewer past purchases also differ in tenure, spend and purchase date, so part of the 0.25 gap may be explained by those features. The final test is whether the flag improves the classification pipeline on the temporal folds.

### Which imputer for the random gaps in `spend_365d`?

As in class 10, Section 5: we can't check a guess for a real gap, so we **make** gaps where we know the answer. We hide 200 known `spend_365d` values in the validation rows, fit each imputer on the training rows, and measure how far each guess lands from the truth, in euros. The split is **by time**: the most recent 20% of train are the validation rows, like the real task (test comes after train). The test is repeated with 3 seeds for the hidden values.

In class, every imputer was about as good as the median, because the COMPAS columns say little about each other. Ours are strongly related to `spend_365d` (`spend_180d`, `purchase_count_365d`, `prior_total_spend`), so the model-based imputers should do better here.

In [11]:
from copy import deepcopy
from time import perf_counter

import numpy as np
import pandas as pd

from src.data import mark_rolling_spend_missing
from src.preprocessing import (
    build_preprocessor,
    predict_spend_eur,
    predictors,
)

field = "spend_365d"
features = config["numeric_features"] + config["categorical_features"]
lookup = customers.reset_index()

# 1. Use the latest 20% of dated purchases for validation.
# Purchases on the same date stay together.
dates = pd.to_datetime(train_raw["purchase_date"])
cutoff = dates.quantile(0.80, interpolation="higher")

history = train_raw.loc[dates < cutoff].copy()
later = train_raw.loc[dates >= cutoff].copy()

X_train = predictors(
    history,
    is_train=True,
    history=history,
    lookup=lookup,
    features=features,
)

# 2. Save valid recorded amounts as the answers, in original euros.
checked, _ = mark_rolling_spend_missing(
    later,
    history=history,
    windows=(365,),
)
truth = checked.set_index("ID")[field]

# Hide annual spend before cleaning so exact recovery gets the first attempt.
X_hidden = predictors(
    later.assign(spend_365d=np.nan),
    is_train=False,
    history=history,
    lookup=lookup,
    features=features,
)

# Eligible purchases have a known answer and still need an imputer.
eligible = X_hidden.index[
    X_hidden[field].isna()
    & truth.reindex(X_hidden.index).notna()
]

if len(eligible) == 0:
    raise ValueError(
        "No known validation amounts still need imputation after masking."
    )

# All imputers are scored on exactly the same purchases.
if len(eligible) > 200:
    masks = [
        np.random.default_rng(seed).choice(
            eligible,
            200,
            replace=False,
        )
        for seed in (0, 1, 2)
    ]
else:
    masks = [eligible]

# 3. Fit each full preprocessor once. Only the imputation strategy changes.
results = []

for strategy in ("median", "mean", "knn", "iterative", "random_forest"):
    settings = deepcopy(config)
    settings["imputation"]["numeric_strategy"] = strategy

    start = perf_counter()
    preprocessor = build_preprocessor(settings).fit(X_train)
    fit_seconds = perf_counter() - start

    predicted = predict_spend_eur(
        preprocessor,
        X_hidden.loc[eligible],
        field,
        settings,
    )

    errors = (predicted - truth.loc[eligible]).abs()

    if not np.isfinite(errors).all():
        raise ValueError(
            f"{strategy} produced a non-finite prediction."
        )

    for ids in masks:
        results.append({
            "imputer": strategy,
            "mean abs error (EUR)": errors.loc[ids].mean(),
            "median abs error (EUR)": errors.loc[ids].median(),
            "fit seconds": fit_seconds,
        })

print(
    f"Validation starts {cutoff.date()} | "
    f"{len(X_train):,} training purchases | "
    f"{len(later):,} validation purchases"
)
print(
    f"{len(eligible):,} eligible purchases | "
    f"{len(masks)} sample(s) of {len(masks[0])} purchases"
)

imputer_test = (
    pd.DataFrame(results)
    .groupby("imputer", sort=False)
    .mean()
)

display(
    imputer_test
    .sort_values("mean abs error (EUR)")
    .round(2)
)

Validation starts 2023-06-17 | 5,171 training purchases | 1,298 validation purchases
853 eligible purchases | 3 sample(s) of 200 purchases


,mean abs error (EUR),median abs error (EUR),fit seconds
imputer,,,
random_forest,1930.65,773.21,7.10
knn,3175.67,1458.82,0.06
iterative,3897.97,1397.69,0.27
median,7032.70,3502.36,0.07
mean,7866.10,4737.97,0.06


### Key insight - Section 3

**Findings**

- **Some gaps follow directly from the customer’s purchase history.** In the cleaned training data, 603 purchases had no earlier purchase, 1,048 had fewer than two, and 1,413 had fewer than three. These groups overlap. For example, `prior_std_gap_days` needs three earlier purchases because it takes two gaps to measure how much the time between purchases varies.
- **Purchase history is informative.** A repeat purchase within 90 days followed 35% of purchases with fewer than two earlier purchases, compared with 60% for those with at least two.
- **Most missing annual spending values could be recovered exactly.** The cleaning rules recovered 187 of 200 missing `spend_365d` values in train and 5 of 40 in test. This leaves **13 training values and 35 test values** for the imputer.
- **Four categorical predictors had missing values in train.** There were 194 missing values in `salesperson`, 195 each in `payment_type` and `warehouse`, and 196 in `transport_method`. Both `commercial_zone` and `document_series` were complete.

**Imputer comparison**

We hid known `spend_365d` values in the chronological validation period. Of these, 853 still needed imputation after applying the exact-recovery rules. We fitted each imputer on earlier purchases using the configured capping, log transformation and scaling. Both methods were scored against the original euro amounts on the same three samples of 200 purchases.


**Decisions**

- Keep `prior_purchase_count` as a predictor. For structural gaps, use `fill_no_history` to assign zero to unavailable amount summaries and measures of variation, `no_history_days` to recency and mean gap, and `no_history_ratio` to `gap_ratio`.
- Apply exact recovery before statistical imputation. **Random forest is the leading candidate for `spend_365d`**. Compare both strategies on repeat-purchase classification and spending regression using the same chronological folds before choosing `numeric_strategy` in `config.yaml`.
- Encode missing categorical values as `"missing"` and retain every test purchase for prediction. Fit preprocessing on each fold’s training predictors only.
- For spending regression, use returning purchases with an observed or verified `spend_next_90d`. A purchase with a known `repeat_purchase_90d` label can still be used for classification when its future spending amount is missing.